In [2]:
!pip install pymorphy3
!pip install pandas


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 5.0 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/9.7 MB 6.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.7 MB 6.8 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.7 MB 6.7 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.7 MB 7.0 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.7 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.7 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 6.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.3 MB 8.9 MB/s eta 0:00:02
   --------- ------------------------------ 2.9/12.3 MB 6.6 MB/s eta 0:00:02
   ----------- ---------------------------- 3.7/12.3 MB 6.4 MB/s eta 0:00:02
   --------------- ---


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install nltk

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------------- ------------ 1.0/1.5 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.7 MB/s eta 0:00:00

   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   -------- ------------------------------- 1/5 [regex]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ------------------------ --------------- 3/5 [click]
   ------------------------ --------------- 3


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import os
import random
import pandas as pd
import string
import pymorphy3
import nltk
nltk.download("stopwords")
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
russian_stopwords = stopwords.words("russian")
morph = pymorphy3.MorphAnalyzer()
from tqdm import tqdm
random.seed(42)
tqdm.pandas()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\obrid\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\obrid\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [146]:
def get_data():
    pairs = os.listdir('./encyclopedic')
    data_pairs = pd.DataFrame({'source':[], 'cos_sim': [], 'target': [], 'level': []})
    
    # объединяем в одну таблицу
    
    for pair in pairs:
      if 'csv' in pair:
        data_pair = pd.read_csv('Encyclopedic/'+pair)
        data_pairs = pd.concat((data_pairs, data_pair))

    data_pairs = data_pairs.drop_duplicates('target')

    search_data = data_pairs[['target']].sample(2000, random_state=42)

    return search_data

### Сама предобработка

In [141]:
def cleaner(text):
  # токенизируем
  sentences = sent_tokenize(text)

  cleaned_sentences = []

  # сохраняем точки для разделения предложений -- удобнее так находить
  for sentence in sentences:
    # чистим от пунктуации
    for punct in string.punctuation+'–«»':
      sentence = sentence.replace(punct, ' ')

    sentence = word_tokenize(sentence)
    # чистим от стоп слов + нижний регистр
    sentence = [word.lower() for word in sentence if word.lower() not in russian_stopwords]

    # лемматизируем
    sentence = [morph.parse(word)[0].normal_form for word in sentence]

    sentence = ' '.join(sentence)
    cleaned_sentences.append(sentence.replace('  ', ' '))

  # возвращаем чистую версию
  result = '. '.join(cleaned_sentences)
  return result

In [147]:
def result_data():
    data = get_data()

    data['cleaned_text'] = data['target'].progress_apply(cleaner)
    data = data.dropna()
    
    data.to_csv('data_infosearch.csv', sep='\t', index=False)
    
    return data    

In [156]:
en_data = result_data() # тут надо разобратся, откуда none в cleaned_text, пресечь на корню

100%|█████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:07<00:00, 271.03it/s]


In [157]:
en_data.head()

,target,cleaned_text
1843,Храм состоит из трех основных частей. Восточна...,храм состоять три основный часть. восточный ча...
457,Один из самых старых театров России. Официальн...,самый старый театр россия. официальный названи...
291,Вскоре Наполеон сделал предложение русскому и...,вскоре наполеон сделать предложение русский им...
2430,Действие трилогии развивается в период с 1914 ...,действие трилогия развиваться период 1914 1920...
278,Даже в самые тяжёлые дни блокады не замирала к...,самый тяжёлый день блокада замирать культурный...


In [158]:
%store en_data

Stored 'en_data' (DataFrame)
